# adaptive-intelligence v2 — Three-Way Comparison
## Traditional RAG vs Adaptive Intelligence vs Adaptive Vectorless

**LLM:** NVIDIA NIM API (free, unlimited, no credit card)  
**Model:** `deepseek-ai/deepseek-r1-distill-llama-70b`  
**Get key:** https://build.nvidia.com → sign up → API keys → copy `nvapi-...`  

**Author:** [@VK_Venkatkumar](https://linkedin.com/in/venkatkumarvk)  
**Library:** [PyPI](https://pypi.org/project/adaptive-intelligence/) | [GitHub](https://github.com/VK-Ant/adaptive-intelligence)  
**Paper:** [ResearchGate](https://www.researchgate.net/publication/405076088)  
**Also:** [llmevalkit](https://pypi.org/project/llmevalkit/) — 61 metrics for LLM evaluation

---

### How to get NVIDIA NIM API key (free, 2 minutes)

1. Go to https://build.nvidia.com
2. Click **Sign In** (top right) → create free NVIDIA Developer account (email only, no credit card)
3. Go to https://build.nvidia.com/settings/api-keys
4. Click **Generate API Key**
5. Copy the key (starts with `nvapi-`)
6. In this Colab: click key icon (left sidebar) → Add secret → Name: `NVIDIA_API_KEY` → paste key

Free: 100+ models, 40 RPM, unlimited usage, no expiry.


## 1. Install

In [ ]:
%%capture
!pip install adaptive-intelligence chromadb openai


## 2. Setup NVIDIA NIM API

In [ ]:
import os, json, time, re
from typing import List, Dict, Any
from collections import Counter
from google.colab import userdata

# ─── API Configuration ─────────────────────────────────
# Priority: NVIDIA NIM (free) > Groq (free) > OpenAI (paid)

API_KEY = None
BASE_URL = None
MODEL = None

providers = [
    ('NVIDIA_API_KEY', 'https://integrate.api.nvidia.com/v1',
     'deepseek-ai/deepseek-r1-distill-llama-70b'),
    ('GROQ_API_KEY', 'https://api.groq.com/openai/v1',
     'llama-3.3-70b-versatile'),
    ('OPENAI_API_KEY', 'https://api.openai.com/v1',
     'gpt-4o-mini'),
]

for key_name, url, model in providers:
    try:
        k = userdata.get(key_name)
        if k:
            API_KEY, BASE_URL, MODEL = k, url, model
            print(f'Using {key_name}')
            print(f'Model: {model}')
            print(f'URL: {url}')
            break
    except Exception:
        continue

if not API_KEY:
    print('No API key found!')
    print('Add NVIDIA_API_KEY in Colab Secrets (key icon, left sidebar)')
    print('Get free key: https://build.nvidia.com/settings/api-keys')


## 3. Create Test Documents

In [ ]:
DOCS = {
    "financial_q3.txt": "NovaTech Q3 2025: Revenue $847M (12.3% YoY). Product $612M (+15.1%), services $235M (+5.8%). Americas $510M, EMEA $220M, APAC $117M. Operating income $127M, margin 15.0% (vs 13.2%). Net income $98M, $2.15/share. EBITDA $168M, margin 19.8%. Cash $1.2B. Q4 guidance: $870-890M revenue, 15.5-16.0% margin.",
    "financial_q2.txt": "NovaTech Q2 2025: Revenue $798M (9.7% YoY). Product $570M, services $228M. Operating income $110M, margin 13.8%. Net income $84M, $1.84/share. EBITDA $149M. 3-week delivery delay from Meridian Semiconductors disrupted APAC. R&D spending $95M (11.9% of revenue) for AI analytics and edge computing.",
    "risk_assessment.txt": "NovaTech Risk 2025: Supply Chain (HIGH) - 65% dependency on Meridian Semiconductors for chips. Secondary sourcing with Pacific Chip Alliance to reduce to 45% by Q2 2026. Cybersecurity (MEDIUM) - 3 intrusion attempts Q2, CyberShield Partners managed security, $12M zero-trust. Regulatory (MEDIUM) - EU AI Act compliance $8-12M. Market (MEDIUM) - AscentTech and Vertex Digital competition, share 28% to 26%. Talent (LOW) - Retention 92% from 88%.",
    "corporate.txt": "NovaTech Structure: CEO Sarah Chen, CFO Marcus Thompson, CTO Dr. Anika Patel, COO James Rodriguez. Subsidiaries: CloudBridge Solutions (100% owned, cloud infrastructure, acquired 2023 for $340M), DataStream Analytics (75%, JV with Apex Capital, real-time analytics), SecureNet Systems (60%, with CyberShield Partners). Key Suppliers: Meridian Semiconductors (65% chip dependency), GlobalFab Manufacturing, TechLogistics. Pacific Chip Alliance is secondary supplier. Competitors: AscentTech Solutions (analytics), Vertex Digital (cloud), Quantum Dynamics (AI infrastructure).",
    "operations.txt": "NovaTech Operations H1 2025: Austin TX facility expanded +18% capacity. New edge computing production line. Manufacturing yield 97.2% (from 95.1%). Pacific Chip Alliance test orders: 50,000 units, 99.1% quality pass rate, full transition Q2 2026. NovaStar Edge Platform launched: 340 enterprise customers, 4.6/5.0 satisfaction, 2.3M events/second. Headcount 12,400 total, added 800 engineers (200 AI/ML). EduTech Foundation partnership trained 150 employees in AI. Carbon emissions reduced 22% YoY. Renewable energy at manufacturing sites reached 68% (from 51%).",
}

DOC_DIR = '/content/docs'
os.makedirs(DOC_DIR, exist_ok=True)
for name, content in DOCS.items():
    with open(os.path.join(DOC_DIR, name), 'w') as f:
        f.write(content)
print(f'{len(DOCS)} documents created in {DOC_DIR}')
for name in DOCS:
    print(f'  {name} ({len(DOCS[name])} chars)')


## 4. Test Queries (20 queries, 5 categories)

In [ ]:
QUERIES = [
    {"q": "What was NovaTech total revenue in Q3 2025?", "cat": "factual", "kw": ["847"]},
    {"q": "What is the EBITDA margin for Q3?", "cat": "factual", "kw": ["19.8"]},
    {"q": "How many employees does NovaTech have?", "cat": "factual", "kw": ["12,400", "12400"]},
    {"q": "What is the manufacturing yield rate?", "cat": "factual", "kw": ["97.2"]},
    {"q": "How is Meridian Semiconductors connected to supply chain risk?", "cat": "relational", "kw": ["65%", "chip"]},
    {"q": "What is CyberShield Partners relationship with NovaTech?", "cat": "relational", "kw": ["managed security"]},
    {"q": "How does Pacific Chip Alliance affect supply chain strategy?", "cat": "relational", "kw": ["secondary", "45%"]},
    {"q": "What entities are connected to DataStream Analytics?", "cat": "relational", "kw": ["Apex Capital", "75%"]},
    {"q": "What are the top risk factors and how are they being mitigated?", "cat": "analytical", "kw": ["supply chain", "cybersecurity"]},
    {"q": "Analyze operational improvements in H1 2025", "cat": "analytical", "kw": ["capacity", "yield"]},
    {"q": "What is the competitive landscape and implications?", "cat": "analytical", "kw": ["AscentTech", "Vertex"]},
    {"q": "Assess R&D investment strategy and impact", "cat": "analytical", "kw": ["AI", "95"]},
    {"q": "Compare financial performance between Q2 and Q3 2025", "cat": "comparative", "kw": ["798", "847"]},
    {"q": "How did operating margin change from Q2 to Q3?", "cat": "comparative", "kw": ["13.8", "15.0"]},
    {"q": "Compare product vs services revenue growth in Q3", "cat": "comparative", "kw": ["15.1", "5.8"]},
    {"q": "How did carbon emissions and renewable energy change YoY?", "cat": "comparative", "kw": ["22%", "68%"]},
    {"q": "Trace Meridian risk to Q2 impact to Pacific Chip mitigation", "cat": "multi_hop", "kw": ["delay", "Pacific Chip"]},
    {"q": "How do subsidiaries position NovaTech against competitors?", "cat": "multi_hop", "kw": ["CloudBridge", "DataStream"]},
    {"q": "Meridian dependency to supply disruption to mitigation to Q4 outlook", "cat": "multi_hop", "kw": ["65%", "870"]},
    {"q": "Full AI strategy: R&D investment, product launch, workforce training, regulatory compliance", "cat": "multi_hop", "kw": ["NovaStar", "EduTech"]},
]

cats = ['factual', 'relational', 'analytical', 'comparative', 'multi_hop']
cat_names = ['Factual', 'Relational', 'Analytical', 'Comparative', 'Multi-hop']
print(f'{len(QUERIES)} queries across {len(cats)} categories')
for c, n in zip(cats, cat_names):
    count = sum(1 for q in QUERIES if q['cat'] == c)
    print(f'  {n}: {count}')


## 5. Setup Three Systems

In [ ]:
from openai import OpenAI
import chromadb
from adaptive_intelligence import AdaptiveAI

# ─── System 1: Traditional RAG ──────────────────────────
class TraditionalRAG:
    def __init__(self):
        self.client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
        self.chroma = chromadb.Client()
        self.col = self.chroma.get_or_create_collection('trad')
        self.chunks = []

    def ingest(self, d):
        cid = 0
        for f in sorted(os.listdir(d)):
            words = open(os.path.join(d, f)).read().split()
            for i in range(0, len(words), 150):
                self.chunks.append({'id': f'c{cid}', 'text': ' '.join(words[i:i+150]), 'src': f})
                cid += 1
        self.col.add(
            ids=[c['id'] for c in self.chunks],
            documents=[c['text'] for c in self.chunks],
            metadatas=[{'src': c['src']} for c in self.chunks])
        print(f'[Traditional RAG] {len(self.chunks)} chunks')

    def ask(self, q):
        t = time.time()
        r = self.col.query(query_texts=[q], n_results=min(5, len(self.chunks)), include=['documents'])
        ctx = '\n---\n'.join(r['documents'][0]) if r['documents'] else ''
        try:
            resp = self.client.chat.completions.create(
                model=MODEL, temperature=0.1, max_tokens=1024,
                messages=[{'role': 'system', 'content': 'Answer from context. Cite sources.'},
                          {'role': 'user', 'content': f'Context:\n{ctx}\n\nQ: {q}'}])
            return {'answer': resp.choices[0].message.content, 'latency': time.time()-t}
        except Exception as e:
            return {'answer': f'Error: {e}', 'latency': time.time()-t}

# ─── System 2: Adaptive Intelligence (vector mode) ─────
adaptive = AdaptiveAI(
    llm_backend='openai', llm_model=MODEL,
    api_key=API_KEY, base_url=BASE_URL,
    domain='financial',
    storage_dir='/content/state_adaptive',
    log_level='WARNING',
)

# ─── System 3: Adaptive Intelligence (vectorless) ──────
adaptive_vl = AdaptiveAI(
    vectorless=True,
    llm_backend='openai', llm_model=MODEL,
    api_key=API_KEY, base_url=BASE_URL,
    domain='financial',
    storage_dir='/content/state_vectorless',
    log_level='WARNING',
)

print('All 3 systems initialized')
print(f'LLM: {MODEL}')


## 6. Ingest Documents

In [ ]:
trad = TraditionalRAG()
trad.ingest(DOC_DIR)

s1 = adaptive.ingest(DOC_DIR)
print(f'[Adaptive] {s1.total_chunks} chunks, {adaptive.graph.node_count} graph nodes, {adaptive.graph.edge_count} edges')

s2 = adaptive_vl.ingest(DOC_DIR)
print(f'[Vectorless] {s2.total_chunks} pages, {adaptive_vl.graph.node_count} graph nodes')


## 7. Run Three-Way Experiment

In [ ]:
def coverage(answer, keywords):
    a = answer.lower()
    return sum(1 for k in keywords if k.lower() in a) / len(keywords) if keywords else 0

R = {'trad': [], 'adapt': [], 'vless': []}

print('Running 20 queries across 3 systems...')
print('=' * 80)

for i, q in enumerate(QUERIES):
    print(f'\n[{i+1}/20] ({q["cat"]}) {q["q"][:60]}...')

    # Traditional RAG
    t = trad.ask(q['q'])
    tc = coverage(t['answer'], q['kw'])
    R['trad'].append({'cat': q['cat'], 'cov': tc, 'answer': t['answer']})

    time.sleep(1.5)  # Rate limit

    # Adaptive Intelligence (vector)
    a = adaptive.ask(q['q'])
    ac = coverage(a.answer, q['kw'])
    R['adapt'].append({
        'cat': q['cat'], 'cov': ac, 'answer': a.answer,
        'strat': a.retrieval_strategy, 'conf': a.confidence,
        'graph': a.retrieval_info.graph_activated if a.retrieval_info else False,
    })

    time.sleep(1.5)

    # Adaptive Intelligence (vectorless)
    v = adaptive_vl.ask(q['q'])
    vc = coverage(v.answer, q['kw'])
    R['vless'].append({
        'cat': q['cat'], 'cov': vc, 'answer': v.answer,
        'strat': v.retrieval_strategy, 'conf': v.confidence,
        'graph': v.retrieval_info.graph_activated if v.retrieval_info else False,
    })

    print(f'  Traditional:  {tc:.0%}')
    print(f'  Adaptive:     {ac:.0%}  [{a.retrieval_strategy}]')
    print(f'  Vectorless:   {vc:.0%}  [{v.retrieval_strategy}]')

    time.sleep(1)

print('\n' + '=' * 80)
print('Experiment complete')


## 8. Results Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def avg_cat(results, cat):
    vals = [r['cov'] for r in results if r['cat'] == cat]
    return sum(vals)/len(vals) if vals else 0

labels = cat_names + ['OVERALL']
t_vals = [avg_cat(R['trad'], c) for c in cats] + [sum(r['cov'] for r in R['trad'])/20]
a_vals = [avg_cat(R['adapt'], c) for c in cats] + [sum(r['cov'] for r in R['adapt'])/20]
v_vals = [avg_cat(R['vless'], c) for c in cats] + [sum(r['cov'] for r in R['vless'])/20]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Three-way bar chart
ax = axes[0]
x = np.arange(len(labels))
w = 0.25
ax.bar(x-w, t_vals, w, label='Traditional RAG', color='#E74C3C', alpha=0.8)
ax.bar(x, a_vals, w, label='Adaptive (vector)', color='#2ECC71', alpha=0.8)
ax.bar(x+w, v_vals, w, label='Adaptive (vectorless)', color='#3498DB', alpha=0.8)
ax.set_ylabel('Keyword Coverage')
ax.set_title('Answer Quality by Category')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8, rotation=15)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=8)
for i in range(len(labels)):
    ax.text(i-w, t_vals[i]+0.02, f'{t_vals[i]:.0%}', ha='center', fontsize=7)
    ax.text(i, a_vals[i]+0.02, f'{a_vals[i]:.0%}', ha='center', fontsize=7)
    ax.text(i+w, v_vals[i]+0.02, f'{v_vals[i]:.0%}', ha='center', fontsize=7)

# Plot 2: Strategy diversity
ax2 = axes[1]
strats_a = Counter(r['strat'] for r in R['adapt'])
strats_v = Counter(r['strat'] for r in R['vless'])
all_s = sorted(set(list(strats_a.keys()) + list(strats_v.keys())))
x2 = np.arange(len(all_s))
ax2.bar(x2-0.15, [strats_a.get(s,0) for s in all_s], 0.3, label='Adaptive', color='#2ECC71', alpha=0.8)
ax2.bar(x2+0.15, [strats_v.get(s,0) for s in all_s], 0.3, label='Vectorless', color='#3498DB', alpha=0.8)
ax2.set_ylabel('Queries')
ax2.set_title('Strategy Diversity')
ax2.set_xticks(x2)
ax2.set_xticklabels([s.replace('_',' ').replace(' + ','\n') for s in all_s], fontsize=7, rotation=30, ha='right')
ax2.legend(fontsize=8)

# Plot 3: Learning curve
ax3 = axes[2]
confs_a = [r['conf'] for r in R['adapt']]
confs_v = [r['conf'] for r in R['vless']]
qnums = list(range(1, 21))
def rolling(vals, w=5):
    return [sum(vals[max(0,i-w+1):i+1])/len(vals[max(0,i-w+1):i+1]) for i in range(len(vals))]
ax3.plot(qnums, rolling(confs_a), color='#2ECC71', linewidth=2, label='Adaptive')
ax3.plot(qnums, rolling(confs_v), color='#3498DB', linewidth=2, label='Vectorless')
ax3.scatter(qnums, confs_a, color='#2ECC71', alpha=0.3, s=30)
ax3.scatter(qnums, confs_v, color='#3498DB', alpha=0.3, s=30)
warmup = adaptive.config.rl.warmup_queries
if warmup <= 20:
    ax3.axvline(x=warmup, color='red', linestyle='--', alpha=0.5)
    ax3.text(warmup+0.3, max(confs_a)*0.95, 'RL active', color='red', fontsize=8)
ax3.set_xlabel('Query #')
ax3.set_ylabel('Confidence')
ax3.set_title('Learning Curve')
ax3.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/content/results.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Results Summary

In [ ]:
print('=' * 75)
print(f'{"Category":<14} {"Trad RAG":>10} {"Adaptive":>10} {"Vectorless":>10} {"Best":>10}')
print('-' * 75)
for cat, name in zip(cats, cat_names):
    tc = avg_cat(R['trad'], cat)
    ac = avg_cat(R['adapt'], cat)
    vc = avg_cat(R['vless'], cat)
    best = 'Adaptive' if ac >= vc else 'Vectorless'
    print(f'{name:<14} {tc:>9.0%} {ac:>9.0%} {vc:>9.0%} {best:>10}')

to = sum(r['cov'] for r in R['trad'])/20
ao = sum(r['cov'] for r in R['adapt'])/20
vo = sum(r['cov'] for r in R['vless'])/20
print('-' * 75)
print(f'{"OVERALL":<14} {to:>9.0%} {ao:>9.0%} {vo:>9.0%}')
print('=' * 75)

print(f'\nStrategies used:')
print(f'  Traditional RAG: 1 (vector_only)')
sa = Counter(r['strat'] for r in R['adapt'])
sv = Counter(r['strat'] for r in R['vless'])
print(f'  Adaptive: {len(sa)} strategies')
for s, c in sorted(sa.items(), key=lambda x: -x[1]):
    print(f'    {s}: {c}/20')
print(f'  Vectorless: {len(sv)} strategies')
for s, c in sorted(sv.items(), key=lambda x: -x[1]):
    print(f'    {s}: {c}/20')

ga = sum(1 for r in R['adapt'] if r.get('graph'))
gv = sum(1 for r in R['vless'] if r.get('graph'))
print(f'\nGraph activated: Trad=0/20 | Adaptive={ga}/20 | Vectorless={gv}/20')


## 10. v2 Feature Demos

### 10a. Vectorless Mode (zero dependencies)

In [ ]:
# Already running as adaptive_vl above
# Key point: no ChromaDB, no embeddings, same RL + graph + evaluation

resp = adaptive_vl.ask('What is the Q3 revenue and EBITDA?')
print('Answer:', resp.answer[:300])
print(f'\nConfidence: {resp.confidence:.0%}')
print(f'Strategy: {resp.retrieval_strategy}')
print(f'\nCitations:')
for c in resp.citations[:3]:
    page = f', Page {c.page}' if c.page else ''
    print(f'  {c.source_document}{page} ({c.confidence:.0%})')


### 10b. Structured Output (JSON)

In [ ]:
resp = adaptive.ask(
    'Extract Q3 financial metrics',
    output_format='json',
    schema={'revenue': 'str', 'operating_margin': 'str', 'ebitda': 'str', 'net_income': 'str'}
)
print('Raw answer:')
print(resp.answer[:400])
print(f'\nParsed structured output: {type(resp.structured)}')
if resp.structured:
    print(json.dumps(resp.structured, indent=2))


### 10c. User Feedback

In [ ]:
resp = adaptive.ask('What are the top 3 risk factors?')
print('Answer:', resp.answer[:200], '...')
print(f'Confidence: {resp.confidence:.0%}')

# Good feedback boosts RL reward for this strategy
adaptive.feedback(resp.query_id, 'good')
print('\nFeedback: good (RL reward boosted for this strategy)')

# Bad feedback penalizes and triggers prompt evolution
resp2 = adaptive.ask('List all subsidiaries')
adaptive.feedback(resp2.query_id, 'bad', reason='Missing ownership percentages')
print('Feedback: bad (RL penalized, prompt evolution triggered)')


## 11. System Dashboards

In [ ]:
print('=== Adaptive Intelligence (Vector Mode) ===')
print(adaptive.dashboard())
print()
print('=== Adaptive Intelligence (Vectorless Mode) ===')
print(adaptive_vl.dashboard())
print()
print('=== Learning Summary ===')
print(adaptive.memory.get_learning_summary())


---

## Version 3 Roadmap

| S.No. | Feature | Category |
|-------|---------|----------|
| 1 | PPO/DQN alongside Thompson Sampling | RL |
| 2 | GraphSAGE embeddings replacing BFS | GNN |
| 3 | Cross-encoder reranking | Retrieval |
| 4 | Multi-query decomposition | Retrieval |
| 5 | Pre-trained domain policies (financial, legal, healthcare) | RL |
| 6 | Transfer learning across deployments | RL |
| 7 | A/B testing (two policies simultaneously) | RL |
| 8 | Multi-modal ingestion (images, charts, audio) | Ingestion |
| 9 | Enterprise connectors (S3, Notion, Confluence) | Platform |
| 10 | llmevalkit compliance integration (HIPAA, GDPR, NIST) | Governance |
| 11 | Plugin API for community connectors | Platform |
| 12 | Scale benchmarks (10K, 50K, 100K docs) | Performance |

---

**adaptive-intelligence v2** | [PyPI](https://pypi.org/project/adaptive-intelligence/) | [GitHub](https://github.com/VK-Ant/adaptive-intelligence) | [Paper](https://www.researchgate.net/publication/405076088)  
**Also:** [llmevalkit](https://pypi.org/project/llmevalkit/) — 61 metrics for LLM evaluation  
**Author:** Venkatkumar Rajan | [@VK_Venkatkumar](https://linkedin.com/in/venkatkumarvk)  
**LLM:** NVIDIA NIM (free) — https://build.nvidia.com
